# SMLM PCF Pipeline: Data Processing

This notebook handles the batch loading, nucleus segmentation, nucleus filtering, and computation of pair correlation functions (PCFs) for SMLM coordinate datasets.

### Processing Steps:
1. **Auto-Pairing**: Automatically scans the input directory to match `*-left*.csv` and `*-right*.csv` files for each field of view (FOV).
2. **Reconstruction & Cellpose Segmentation**: Builds a 2D density image from the high-density channel and segments nuclei.
3. **Nucleus Filtering**: Discards nuclei that are too small or too close to the image border.
4. **PCF Calculation with Rich Progress**: Computes Auto-Left, Auto-Right, and Cross PCF for each valid nucleus sequentially, with a live `rich` progress bar.

### Step 1: Imports and Analysis Parameters

In [17]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from cellpose import models, core, io
from rich.progress import Progress, TextColumn, BarColumn, MofNCompleteColumn, TimeElapsedColumn, TimeRemainingColumn, SpinnerColumn
from rich.console import Console

from smlm_pcf_fft import is_valid_nucleus, filter_points_in_mask, calculate_pcf_fft

console = Console()

# ==========================================
# ANALYSIS PARAMETERS - edit these
# ==========================================

# Directory containing paired left & right dSTORM CSVs
input_dir = '/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-dSTORM/Dox24h'
output_dir = '/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-dSTORM/Dox24h'

# Density Reconstruction & Segmentation
pixel_size       = 100.0  # nm per pixel for density image
sigma_blur       = 5.0    # Gaussian blur sigma in pixels (500 nm)
nuc_diameter     = 80     # Expected nucleus diameter in pixels (~8 µm)

# Nucleus Quality Filters
min_area_pixels     = 6000  # Minimum nucleus area in pixels; smaller = debris/partial
edge_margin_pixels  = 0     # Reject nuclei whose bounding box is within this many pixels of the image edge
min_points_per_nuc  = 500   # Minimum localizations inside a nucleus mask to attempt PCF

# Sliding-ring PCF bins (nm)
r_max_nm          = 1120.0
ringwidth_nm      = 100.0
dr_nm             = 20.0

bin_starts = np.arange(0, r_max_nm - ringwidth_nm, dr_nm)
bin_ends   = bin_starts + ringwidth_nm

os.makedirs(output_dir, exist_ok=True)
console.print('[green]Parameters configured.[/green]')

Parameters configured.

### Step 2: Auto-Pair Left / Right CSV Files

In [18]:
console.print(f'Scanning [bold]{input_dir}[/bold] for left/right pairs...')
left_files = sorted(glob.glob(os.path.join(input_dir, '*left*.csv')))
paired_files = []

for lf in left_files:
    rf = lf.replace('left', 'right')
    if os.path.exists(rf):
        base = os.path.basename(lf)
        # Strip common suffixes to get a clean FOV name
        fov_name = base
        for suffix in ['-cropped-left.csv', '-left.csv', '_left.csv',
                        '-cropped-left', '-left', '_left']:
            fov_name = fov_name.replace(suffix, '')
        paired_files.append((fov_name, lf, rf))

console.print(f'Found [bold]{len(paired_files)}[/bold] paired FOV dataset(s):')
for fov, lf, rf in paired_files:
    console.print(f'  [cyan]{fov}[/cyan]')
    console.print(f'    Left  → {os.path.basename(lf)}')
    console.print(f'    Right → {os.path.basename(rf)}')

Scanning 
/Volumes/guttman/users/gmgao/Imaging_ProcessedData/SPEN/20260413_ONI-gmgao-SPEN_H3K27ac_dSTORM_SACD/processed-dSTOR
M/Dox24h for left/right pairs...

Found 3 paired FOV dataset(s):

TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1

Left  → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1-cropped-left.csv

Right → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1-cropped-right.csv

...

Left  → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-11-cropped-left.csv

Right → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-11-cropped-right.csv

TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2

Left  → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2-cropped-left.csv

Right → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2-cropped-right.csv

### Step 3: Segmentation and Nucleus Filtering

In [19]:
# Load Cellpose model once
console.print('Loading Cellpose model...')
gpu_avail = core.use_gpu()
model = models.CellposeModel(gpu=gpu_avail)
console.print(f'  GPU: {gpu_avail}')

# List of tasks: one entry per valid nucleus
nucleus_tasks = []

for fov_name, lf, rf in paired_files:
    console.rule(f'[bold cyan]{fov_name}[/bold cyan]')

    # Load coordinates
    console.print('  Loading coordinates...')
    df_left  = pd.read_csv(lf)
    df_right = pd.read_csv(rf)
    pts_left  = df_left[['x [nm]', 'y [nm]']].values
    pts_right = df_right[['x [nm]', 'y [nm]']].values

    # Reconstruct density from high-density (right) channel
    x_r, y_r = pts_right[:, 0], pts_right[:, 1]
    x_min, x_max = x_r.min(), x_r.max()
    y_min, y_max = y_r.min(), y_r.max()

    x_edges = np.arange(x_min, x_max + pixel_size, pixel_size)
    y_edges = np.arange(y_min, y_max + pixel_size, pixel_size)
    H, _, _ = np.histogram2d(y_r, x_r, bins=(y_edges, x_edges))
    smoothed   = gaussian_filter(H, sigma=sigma_blur)
    normalized = smoothed / smoothed.max() if smoothed.max() > 0 else smoothed

    # Cellpose segmentation
    console.print('  Running Cellpose segmentation...')
    img_in = normalized[np.newaxis].astype(np.float32)
    masks_nuclei, _, _ = model.eval(
        img_in,
        batch_size=32,
        diameter=nuc_diameter,
        flow_threshold=0.5,
        cellprob_threshold=0.0,
        normalize={'tile_norm_blocksize': 0}
    )

    # Save mask
    mask_path = os.path.join(output_dir, f'{fov_name}_masks_nuclei.tif')
    io.imsave(mask_path, masks_nuclei.astype(np.int32))
    console.print(f'  Mask saved → {os.path.basename(mask_path)}')

    # Filter nuclei
    labels = np.unique(masks_nuclei)
    labels = labels[labels != 0]

    valid_labels, skipped_labels = [], []
    for label in labels:
        if is_valid_nucleus(masks_nuclei, label, min_area_pixels, edge_margin_pixels):
            valid_labels.append(label)
        else:
            skipped_labels.append(label)

    console.print(f'  [green]{len(valid_labels)} valid nuclei[/green] | '
                  f'[red]{len(skipped_labels)} rejected[/red] (too small or near edge)')

    # Build cropped mask tasks
    for label in valid_labels:
        y_idx, x_idx = np.where(masks_nuclei == label)
        pad = 15
        y0 = max(0, y_idx.min() - pad)
        y1 = min(masks_nuclei.shape[0], y_idx.max() + pad + 1)
        x0 = max(0, x_idx.min() - pad)
        x1 = min(masks_nuclei.shape[1], x_idx.max() + pad + 1)

        nuc_mask     = (masks_nuclei[y0:y1, x0:x1] == label)
        nuc_x_min    = x_min + x0 * pixel_size
        nuc_y_min    = y_min + y0 * pixel_size

        nucleus_tasks.append(dict(
            fov_name=fov_name,
            label=int(label),
            pts_left=pts_left,
            pts_right=pts_right,
            nuc_mask=nuc_mask,
            pixel_size=pixel_size,
            nuc_x_min=nuc_x_min,
            nuc_y_min=nuc_y_min,
        ))

console.print(f'\n[bold green]Total nuclei queued for PCF: {len(nucleus_tasks)}[/bold green]')

Loading Cellpose model...

GPU: True

─────────────────────────────── TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1 ────────────────────────────────

Loading coordinates...

Running Cellpose segmentation...

...

Loading coordinates...

Running Cellpose segmentation...

Mask saved → TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2_masks_nuclei.tif

11 valid nuclei | 2 rejected (too small or near edge)

Total nuclei queued for PCF: 33

### Step 4: PCF Calculation with Rich Progress Bar

Runs sequentially — each nucleus requires 3 KDTree passes (Auto-Left, Auto-Right, Cross).  
A `rich` progress bar tracks overall completion.

In [20]:
results = []

with Progress(
    SpinnerColumn(),
    TextColumn('[progress.description]{task.description}'),
    BarColumn(),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    TimeRemainingColumn(),
    console=console,
    transient=False,
) as progress:

    nuc_task = progress.add_task('[cyan]Nuclei PCF', total=len(nucleus_tasks))

    for task in nucleus_tasks:
        fov    = task['fov_name']
        label  = task['label']
        progress.update(nuc_task, description=f'[cyan]{fov} | nucleus {label}')

        pts_l = filter_points_in_mask(
            task['pts_left'],  task['nuc_mask'],
            task['nuc_x_min'], task['nuc_y_min'], task['pixel_size'])
        pts_r = filter_points_in_mask(
            task['pts_right'], task['nuc_mask'],
            task['nuc_x_min'], task['nuc_y_min'], task['pixel_size'])

        if len(pts_l) < min_points_per_nuc or len(pts_r) < min_points_per_nuc:
            progress.console.print(
                f'  [yellow]Skipped {fov} nuc {label} '
                f'(L={len(pts_l)}, R={len(pts_r)} < {min_points_per_nuc})[/yellow]')
            results.append({'fov': fov, 'label': label, 'status': 'skipped'})
            progress.advance(nuc_task)
            continue

        # --- Auto-Left ---
        progress.update(nuc_task, description=f'[cyan]{fov} | nuc {label} — Auto-Left')
        G_left, r_centers, _, _ = calculate_pcf_fft(
            pts_l, None, task['nuc_mask'], task['pixel_size'],
            task['nuc_x_min'], task['nuc_y_min'], bin_starts, bin_ends, mode='auto')

        # --- Auto-Right ---
        progress.update(nuc_task, description=f'[cyan]{fov} | nuc {label} — Auto-Right')
        G_right, _, _, _ = calculate_pcf_fft(
            pts_r, None, task['nuc_mask'], task['pixel_size'],
            task['nuc_x_min'], task['nuc_y_min'], bin_starts, bin_ends, mode='auto')

        # --- Cross ---
        progress.update(nuc_task, description=f'[cyan]{fov} | nuc {label} — Cross')
        G_cross, _, _, _ = calculate_pcf_fft(
            pts_l, pts_r, task['nuc_mask'], task['pixel_size'],
            task['nuc_x_min'], task['nuc_y_min'], bin_starts, bin_ends, mode='cross')

        # Save per-nucleus CSV
        csv_path = os.path.join(output_dir, f'{fov}_nucleus_{label}_pcf.csv')
        pd.DataFrame({
            'r_center_nm':  r_centers,
            'bin_start_nm': bin_starts,
            'bin_end_nm':   bin_ends,
            'G_auto_left':  G_left,
            'G_auto_right': G_right,
            'G_cross':      G_cross,
        }).to_csv(csv_path, index=False)

        results.append({'fov': fov, 'label': label, 'status': 'success', 'csv': csv_path})
        progress.console.print(f'  [green]✓[/green] {fov} nuc {label} → {os.path.basename(csv_path)}')
        progress.advance(nuc_task)

n_ok   = sum(1 for r in results if r['status'] == 'success')
n_skip = sum(1 for r in results if r['status'] == 'skipped')
console.print(f'\n[bold green]Done![/bold green] {n_ok} nuclei processed, {n_skip} skipped.')

Output()

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1 nuc 1 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1_nucleus_1_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1 nuc 2 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1_nucleus_2_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1 nuc 3 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1_nucleus_3_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1 nuc 4 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-1_nucleus_4_pcf.csv

...

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2 nuc 9 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2_nucleus_9_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2 nuc 10 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2_nucleus_10_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2 nuc 12 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2_nucleus_12_pcf.csv

✓ TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2 nuc 13 → 
TXSHA-Dox24h-SPEN_JF549_H3K27ac_DL650-dSTORM-FOV-2_nucleus_13_pcf.csv

Done! 33 nuclei processed, 0 skipped.